## Fetch and Do KRX Stocks
KRX, KOSDAQ, KOSPI, 여러 종목

In [2]:
# _tsa_00.py
# time series stuff, basic
# 그리기
# 시계열, 추세, 계절성 조정, 이동평균, 이런 것들.
# 자기공분산, 자기상관, 단위근
# ARIMA 중 AR(p)
# 기타... 단순한 내용만 간략하게.

# 코랩 추가.
!pip install yfinance
#!pip install pykrx   # 이건 스크래핑, 무단 자료조회 모듈임. 스킵.
!pip install finance-datareader

import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf                       ## NYSE 데이터 모듈
import FinanceDataReader as fdr             ## KOSPI, NYSE
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭.
from statsmodels.tsa.ar_model import AutoReg


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 861.2 kB/s eta 0:00:00


### 종목 코, 영문 약칭

In [7]:
# 별도로 종목 리스트 받음.
# 20260901_krx_list 1. 라이브러리 설치 (코랩의 경우 맨 앞에 ! 추가)
#!pip install yfinance
# import yfinance as yf

krx_list_url_1 = 'https://github.com/bahn28/salad/blob/main/kospi_stock_list_20260901.csv?raw=true'
df_krx_lst_1 = pd.read_csv(krx_list_url_1)
df_krx_lst_1 = df_krx_lst_1[['srtnCd', 'itmsNm', 'mrktCtg', 'mrktTotAmt']]
df_krx_lst_1.info() #head()   # 여기에 회사이름 등이 모두 한글로... 영문으로 바꾸는 아이디어. 매치.

krx_list_url_2 = 'https://github.com/bahn28/salad/blob/main/krx_list_all_engconame.csv?raw=true'
df_krx_lst_2 = pd.read_csv(krx_list_url_2)
df_krx_lst_2.info() #head()  여기 단축코드가 저기 srtnCd임. 이걸 매치시켜야.

df_krx_lst = pd.merge(df_krx_lst_1, df_krx_lst_2, on='srtnCd')  # 뭐가 맞아야 합쳐지지. 아직은 아님.
df_krx_lst.info()

## 이하는 셀렉션.
#df_krx_lst.info() # 2873 by 15, 영어 코드로 작성된걸 받자. 검색 재실행.
# 코드, 종목명 짝을 적어놓차.
# KOSDAQ list
df_ksd = df_krx_lst[(df_krx_lst['mrktCtg']=='KOSDAQ') & (df_krx_lst['mrktTotAmt']>10**12.5) ]
#df_ksd.info()  # 2873 by 15 -> 1822  -> 76 -> 20
df_ksd
# KOSPI list
df_ksp = df_krx_lst[(df_krx_lst['mrktCtg']=='KOSPI') & (df_krx_lst['mrktTotAmt']>10**13) ]
#df_ksp.info()  # 67 by 15
df_ksp
df_krx = pd.concat([df_ksd, df_ksp])  # 위-아래 붙이기
df_krx.info()
df_krx.rename(columns={'srtnCd': 'ticker', 'itmsNm': 'company_kr'}, inplace=True)
df_krx.sort_values(by='mrktTotAmt', inplace=True, ascending=False)
df_krx

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2873 entries, 0 to 2872
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   srtnCd      2873 non-null   object
 1   itmsNm      2873 non-null   object
 2   mrktCtg     2873 non-null   object
 3   mrktTotAmt  2873 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 89.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2874 entries, 0 to 2873
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   표준코드     2874 non-null   object
 1   단축코드     2874 non-null   object
 2   한글 종목명   2874 non-null   object
 3   한글 종목약명  2874 non-null   object
 4   영문 종목명   2874 non-null   object
 5   상장일      2874 non-null   object
 6   시장구분     2874 non-null   object
 7   증권구분     2874 non-null   object
 8   소속부      1931 non-null   object
 9   주식종류     2874 non-null   object
 10  액면가      2874 non-null   object
 11  상장주식수    2

KeyError: 'srtnCd'

### 개별 종목, 가격, 거래량

In [4]:
tickers = df_krx['ticker'][:12]
# 1. 삼성전자(005930)의 2025년부터 현재까지의 주가 데이터 수집
#tickers = ['005930', '000660']    # 삼성전자 코드, 티커.
df_dat = fdr.DataReader(tickers, '2023-01-01')  # 삼성전자
#ticker_2 = '000660'  # SK하이익스
#df_dat_2 = fdr.DataReader(ticker_2, '2023-01-01')  # 하이닉스

# # 2. 가격(종가) 데이터 읽기
# # 최신 버전 FinanceDataReader의 주가 컬럼명은 'Close'입니다.

#prc_cls = df_dat['Close']
#prc_cls_hy = df_dat_2['Close']

# # 3. 거래량(볼륨) 데이터 읽기
#vlm = df_dat['Volume']
#vlm_2 = df_dat_2['Volume']

# # 4. 상위 5개 데이터 결합해서 눈으로 확인하기
# print(df_dat[['Close', 'Volume']].head())
# print(df_dat.head(15) )
df_dat.info()
df_dat.head()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 901 entries, 2023-01-02 to 2026-09-10
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   005930  901 non-null    int64
 1   000660  901 non-null    int64
 2   005935  901 non-null    int64
 3   402340  901 non-null    int64
 4   009150  901 non-null    int64
 5   373220  901 non-null    int64
 6   005380  901 non-null    int64
 7   207940  901 non-null    int64
 8   028260  901 non-null    int64
 9   032830  901 non-null    int64
 10  105560  901 non-null    int64
 11  012450  901 non-null    int64
dtypes: int64(12)
memory usage: 91.5 KB


,005930,000660,005935,402340,009150,373220,005380,207940,028260,032830,105560,012450
Date,,,,,,,,,,,,
2023-01-02,55500,75700,50800,32900,132500,446000,157000,1217132,111500,70400,47600,74972
2023-01-03,55400,75600,50500,32000,139500,440500,159000,1181810,111000,69400,49050,72732
2023-01-04,57800,81000,52100,33450,143500,443000,160500,1168565,115500,70900,50500,73139
2023-01-05,58200,81400,53600,33400,145000,433500,159000,1189169,113500,71000,53900,68046
2023-01-06,59000,83100,53700,34050,143500,444000,159500,1189169,114500,72400,56700,68249


In [5]:
# daily stock prices and trading volumes
# 변수로 받는 방법.
yymmdd = df_dat.index   # 자료를 불러올 때, 자동으로 기준 인덱스로 설정된다고 함.
yymmdd = pd.to_datetime( yymmdd )

stck_prc = df_dat['Close']
stck_prc_dff = stck_prc.diff()
stck_prc_pct = stck_prc.pct_change()

stck_vlm = df_dat['Volume']
stck_vlm_dff = stck_vlm.diff()
stck_vlm_pct = stck_vlm.pct_change()

stck_prc_2 = df_dat_2['Close']
stck_prc_dff_2 = stck_prc_2.diff()
stck_prc_pct_2 = stck_prc_2.pct_change()

stck_vlm_2 = df_dat_2['Volume']
stck_vlm_dff_2 = stck_vlm_2.diff()
stck_vlm_pct_2 = stck_vlm_2.pct_change()


KeyError: 'Close'

### plots, daily price level

In [ ]:

plt.figure(figsize=(8, 5))
#plt.subplot(figsize=(8,5  ))
plt.plot(yymmdd, stck_prc, color='blue', linewidth=1)
plt.title(f" Closing Prices of {ticker} ")

plt.figure(figsize=(8, 5))
#plt.subplot(figsize=(8,5  ))
plt.plot(yymmdd, stck_prc_2, color='blue', linewidth=1)
plt.title(f" Closing Prices of {ticker_2} ")


### plots, daily price/volume changes

In [ ]:

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
#axes[0].plot(yymmdd, stck_prc, color='blue', linewidth=0.5)
axes[0].plot(yymmdd, stck_prc_pct, color='blue', linewidth=0.5)
axes[1].plot(yymmdd, stck_vlm, color='red', linewidth=0.5)
# axes[0].set_title(f" Closing Prices of {ticker} ")
axes[0].set_title(f" Daily Price Changes of {ticker} ")
axes[1].set_title(f" Daily Trading Volumes of {ticker} ")
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
#axes[0].plot(yymmdd, stck_prc, color='blue', linewidth=0.5)
axes[0].plot(yymmdd, stck_prc_pct_2, color='blue', linewidth=0.5)
axes[1].plot(yymmdd, stck_vlm_2, color='red', linewidth=0.5)
# axes[0].set_title(f" Closing Prices of {ticker} ")
axes[0].set_title(f" Daily Price Changes of {ticker_2} ")
axes[1].set_title(f" Daily Trading Volumes of {ticker_2} ")
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))


### histogram, pct changes

In [ ]:

plt.figure(figsize=(8, 5))
plt.hist(stck_prc_pct,
         color='blue',
         linewidth=1,
         bins = 16,
         facecolor='None',
         edgecolor='black',
         density=True
         )
plt.title(f" distribution of daily percentage change of {ticker} ")
#plt.show()

plt.hist(stck_prc_pct_2,
         color='blue',
         linewidth=1,
         bins = 16,
         facecolor='None',
         edgecolor='blue',
         density=True
         )
plt.title(f" distribution of daily percentage change of {ticker_2} ")
#plt.show()